# EasyVitessce Example: SpatialData-Plot with MERFISH dataset

## Downloading and importing necessary packages

By default, interactive plots are enabled upon importing easy_vitessce. This notebook aims to demonstrate the transition between static and interactive plots, so the interactive plots are initially turned off.

In [ ]:
!pip install easy_vitessce
!pip install spatialdata
!pip install spatialdata_plot

In [1]:
import easy_vitessce as ev 
import spatialdata as sd
import spatialdata_plot
from os.path import join

/Users/mkeller/research/dbmi/vitessce/easy_vitessce/.venv/lib/python3.12/site-packages/dask/dataframe/__init__.py:31: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  warnings.warn(
/Users/mkeller/research/dbmi/vitessce/easy_vitessce/.venv/lib/python3.12/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


In [2]:
from vitessce.data_utils import (
    sdata_morton_sort_points,
    sdata_points_process_columns,
    sdata_points_write_bounding_box_attrs,
    sdata_points_modify_row_group_size,
    sdata_morton_query_rect,
)

In [3]:
# Disable interactive plots
ev.configure_plots(enable_plots=["spatialdata-plot"])

## Download the data

In [4]:
import os
from os.path import join, isfile, isdir
from urllib.request import urlretrieve
import zipfile

In [5]:
data_dir = "data"
zip_path = join(data_dir, "xenium_rep1_io.spatialdata.zarr.zip")
sdata_path = join(data_dir, "xenium_rep1_io.spatialdata.zarr")

In [6]:
if not isdir(sdata_path):
    if not isfile(zip_path):
        os.makedirs(data_dir, exist_ok=True)
        urlretrieve('https://s3.embl.de/spatialdata/spatialdata-sandbox/xenium_rep1_io.zip', zip_path)
    with zipfile.ZipFile(zip_path,"r") as zip_ref:
        zip_ref.extractall(data_dir)
        os.rename(join(data_dir, "data.zarr"), sdata_path)

## Read the data

In [7]:
sdata = sd.read_zarr(sdata_path)
sdata

version mismatch: detected: RasterFormatV02, requested: FormatV04
/Users/mkeller/research/dbmi/vitessce/easy_vitessce/.venv/lib/python3.12/site-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
version mismatch: detected: RasterFormatV02, requested: FormatV04


SpatialData object, with associated Zarr store: /Users/mkeller/research/dbmi/vitessce/easy_vitessce/docs/notebooks/data/xenium_rep1_io.spatialdata.zarr
├── Images
│     ├── 'morphology_focus': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213)
│     └── 'morphology_mip': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213)
├── Points
│     ├── 'transcripts': DataFrame with shape: (<Delayed>, 8) (3D points)
│     └── 'transcripts_with_morton_codes': DataFrame with shape: (<Delayed>, 12) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (167780, 1) (2D shapes)
│     └── 'cell_circles': GeoDataFrame shape: (167780, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (167780, 313)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), morphology_mip (Images), transcripts (Points), transcripts_with_morton_codes (Points), cell_boundari

## Make the points ready for tiled access

References:
- https://vitessce.io/docs/data-troubleshooting/#points
- https://github.com/vitessce/vitessce-python/blob/main/docs/notebooks/spatial_data_xenium_morton.ipynb

In [8]:
if "transcripts_with_morton_codes" not in sdata.points:
    sdata = sdata_morton_sort_points(sdata, "transcripts")
    
    # Add feature_index column to dataframe, and reorder columns so that feature_name (dict column) is the rightmost column.
    ddf = sdata_points_process_columns(sdata, "transcripts", var_name_col="feature_name", table_name="table")
    
    sdata["transcripts_with_morton_codes"] = ddf
    sdata.write_element("transcripts_with_morton_codes")
    
    sdata_points_write_bounding_box_attrs(sdata, "transcripts_with_morton_codes")
    
    sdata_points_modify_row_group_size(sdata, "transcripts_with_morton_codes", row_group_size=25_000)

## Static plotting

In [9]:
sdata

SpatialData object, with associated Zarr store: /Users/mkeller/research/dbmi/vitessce/easy_vitessce/docs/notebooks/data/xenium_rep1_io.spatialdata.zarr
├── Images
│     ├── 'morphology_focus': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213)
│     └── 'morphology_mip': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213)
├── Points
│     ├── 'transcripts': DataFrame with shape: (<Delayed>, 8) (3D points)
│     └── 'transcripts_with_morton_codes': DataFrame with shape: (<Delayed>, 12) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (167780, 1) (2D shapes)
│     └── 'cell_circles': GeoDataFrame shape: (167780, 2) (2D shapes)
└── Tables
      └── 'table': AnnData (167780, 313)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), morphology_mip (Images), transcripts (Points), transcripts_with_morton_codes (Points), cell_boundari

In [11]:
sdata.points["transcripts_with_morton_codes"]

,x,y,z,cell_id,overlaps_nucleus,transcript_id,qv,x_uint,y_uint,morton_code_2d,feature_name_codes,feature_name
npartitions=8,,,,,,,,,,,,
,float32,float32,float32,int32,uint8,uint64,float32,uint32,uint32,uint32,int32,category[unknown]
,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...


In [12]:
sdata.points["transcripts_with_morton_codes"].attrs

{'transform': {'global': Scale (x, y, z)
      [4.70588235 4.70588235 1.        ]},
 'spatialdata_attrs': {'feature_key': 'feature_name',
  'instance_key': 'cell_id'}}

In [10]:
vw = (
    sdata
        .pl.render_images(element="morphology_focus")
        .pl.render_shapes(element="cell_boundaries")
        .pl.render_points(element="transcripts_with_morton_codes", color="feature_name", groups=["ERBB2"], palette=["red"])
        .pl.show()
)
vw

VitessceWidget(js_dev_mode=True, uid='eed7')

In [ ]:
# add another code block for the mouse liver dataset? 

## Activating interactive plots

In [ ]:
# Enable interactive plots
ev.configure_plots(enable_plots=["spatialdata-plot"])

## Interactive plotting

In [ ]:
sdata.pl.render_images(element="rasterized").pl.render_shapes(element="cells", color="Acta2").pl.show()